# 05_02 Training your own: can 800 support tickets teach a network what a town is?

GloVe learned from six billion words. Kittiwake has about 25,000: every ticket and review it owns. In
this notebook you build the skip-gram model of the chapter in PyTorch, one piece at a time, train it on
that text in about a quarter of a minute, and find out whether so little text is enough to learn that
Gullhaven and Saltmoor are the same kind of thing. The book did this with the gensim library; writing it
yourself takes about thirty lines and removes all the mystery.

**How this notebook works.** Every notebook in this course has the same rhythm:

1. **Recall.** Answer from memory before you look anything up. `ask()` tells you at once whether you were right.
2. **Predict, then run.** Before a cell with a surprise in it, write your prediction into `guess()`. The next cell runs the code and `reveal()` compares.
3. **Worked example, then your turn.** One example is done in full; the next, near-identical one has lines marked `# YOUR CODE HERE`.
4. **Check.** A `check_...()` cell tests what you saved, exactly as the checkpoint will, and says what to fix.

Run cells in order with **Shift+Enter**. If you get lost, **Kernel, Restart Kernel and Run All Cells** starts clean.

Running this in Google Colab? This cell sets it up; in CourseLabs it does nothing.

In [ ]:
# Colab setup. In a CourseLabs session this cell does nothing.
import os, sys
if "google.colab" in sys.modules:
    import importlib, importlib.util, subprocess
    LAB, REPO = "lab-nlp-05-words-as-points-in-space", "/content/nlp-course"
    if not os.path.isdir(REPO):
        subprocess.run(["git", "clone", "-q", "--depth", "1", "https://github.com/fenago/nlp-course.git", REPO], check=True)
    os.chdir(f"{REPO}/{LAB}")
    if not os.path.exists("data"):
        os.symlink("../data", "data")
    os.makedirs("out", exist_ok=True)
    os.environ["NLPLAB_DATA"] = f"{REPO}/data"
    sys.path.insert(0, os.getcwd())
    PIP = {'torch': 'torch',
           'pandas': 'pandas',
           'numpy': 'numpy'}
    missing = [spec for mod, spec in PIP.items() if importlib.util.find_spec(mod) is None]
    if missing:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", *missing], check=True)
        importlib.invalidate_caches()
    print(f"Ready: {LAB} and its data are in {os.getcwd()}; installed {len(missing)} package(s).")
elif not os.path.isdir("/opt/nlplab/data") and os.path.isdir("data"):
    # A downloaded copy on your own computer: the helpers read data/ from here.
    os.environ["NLPLAB_DATA"] = os.path.abspath("data")

In [ ]:
import collections
import json
import os
import time
import numpy as np
import torch
from w2vtools import kittiwake_sentences, nearest
from nlpcheck import ask, guess, reveal, check_05_02

sentences = kittiwake_sentences()
print(len(sentences), "tickets and reviews,", sum(map(len, sentences)), "words")
print(sentences[0])

## 1. Recall

From the last notebook.

**r3.** What word did `king - man + woman` land nearest? (one word)

**r4.** Why are *hot* and *cold* close in GloVe? (a) GloVe was trained badly, (b) they are spelled
alike, (c) they appear in the same contexts

In [ ]:
ask("r3", "")
ask("r4", "")

## 2. The training data is pairs

Skip-gram never sees a sentence as a whole. It sees **pairs**: a centre word, and one word near it.
With a window of 2, every word is paired with up to two words on each side. Here are the pairs of one
short sentence, worked in full. Predict first how many there will be for these five words.

In [ ]:
example = ["calls", "drop", "in", "port", "ember"]
guess("pairs_in_example", None)   # a number

In [ ]:
window = 2
example_pairs = []
for i, centre in enumerate(example):
    for j in range(max(0, i - window), min(len(example), i + window + 1)):
        if j != i:
            example_pairs.append((centre, example[j]))
print(example_pairs)
reveal("pairs_in_example", len(example_pairs))

Fourteen, not twenty: the words at each end have fewer neighbours. The two `range` bounds clip the
window at the edges of the sentence, and `j != i` stops a word pairing with itself.

The model works with numbers, so each word gets an id: its position in a vocabulary of the words that
appear at least three times (rarer ones have too few pairs to learn from).

In [ ]:
counts = collections.Counter(w for s in sentences for w in s)
vocab = [w for w, c in counts.most_common() if c >= 3]
index = {w: i for i, w in enumerate(vocab)}
print(len(vocab), "words in the vocabulary; the most common:", vocab[:12])

**Your turn:** finish `make_pairs`. It does exactly what the worked example did, for every sentence,
using ids instead of words and skipping any word not in `index`. It returns a list of `(centre_id,
context_id)` tuples.

In [ ]:
def make_pairs(sentences, index, window):
    pairs = []
    for s in sentences:
        ids = [index[w] for w in s if w in index]
        # YOUR CODE HERE: the same double loop as the worked example, over ids
    return pairs

pairs = make_pairs(sentences, index, window=3)
print(len(pairs), "pairs with a window of 3; the first five:", pairs[:5])

With `make_pairs` right, a window of 3 gives 137,760 pairs from these 25,500 words: every word is the
centre of up to six pairs.

## 3. The model: two tables of numbers and a dot product

Skip-gram's network is as small as a network can be. It has two **embedding tables**, each with one
row of 32 numbers per word: one table for words as centres, one for words as contexts. The model's score
for a pair is the dot product of the centre word's row and the context word's row. Training makes that
score high for pairs that really occur.

But a model that only ever sees true pairs can cheat by making every score high. **Negative sampling**
stops it: for each true pair, draw a few random words (here 5) and train their scores to be low. The
random words are drawn in proportion to their frequency raised to the power 0.75, which gives rare
words a slightly better chance than their frequency alone. The loss below is the standard one: the log of
the sigmoid of the true score, plus the log of the sigmoid of minus each random score, negated so that
lower is better.

Training takes about ten seconds in a session: five passes over the 137,760 pairs, in batches of
1,024.

In [ ]:
torch.manual_seed(0)
dim, negatives = 32, 5
centre_emb = torch.nn.Embedding(len(vocab), dim)
context_emb = torch.nn.Embedding(len(vocab), dim)
torch.nn.init.uniform_(centre_emb.weight, -0.5 / dim, 0.5 / dim)
torch.nn.init.zeros_(context_emb.weight)
noise = torch.tensor([counts[w] for w in vocab], dtype=torch.float) ** 0.75
opt = torch.optim.Adam(list(centre_emb.parameters()) + list(context_emb.parameters()), lr=0.01)
P = torch.tensor(pairs if pairs else [(0, 0)])

start = time.time()
for epoch in range(5):
    order, total = torch.randperm(len(P)), 0.0
    for b in range(0, len(P), 1024):
        batch = P[order[b:b + 1024]]
        c, o = batch[:, 0], batch[:, 1]
        neg = torch.multinomial(noise, len(c) * negatives, replacement=True).view(len(c), negatives)
        vc = centre_emb(c)
        pos_score = (vc * context_emb(o)).sum(1)                                # true pairs
        neg_score = torch.bmm(context_emb(neg), vc.unsqueeze(2)).squeeze(2)     # random pairs
        loss = -(torch.nn.functional.logsigmoid(pos_score).mean()
                 + torch.nn.functional.logsigmoid(-neg_score).sum(1).mean())
        opt.zero_grad(); loss.backward(); opt.step()
        total += loss.item() * len(c)
    print(f"epoch {epoch + 1}: loss {total / len(P):.3f}")
print(f"trained in {time.time() - start:.0f} s")

E = centre_emb.weight.detach().numpy().copy()
E /= np.linalg.norm(E, axis=1, keepdims=True)

The loss falls from about 2.8 to about 1.6 and flattens: the model has learned most of what these pairs
can teach it. Four lines of that cell deserve a second look. `centre_emb(c)` looks up the rows for a batch
of centre words. `(vc * context_emb(o)).sum(1)` is a dot product per pair. `torch.bmm` does the same for
the five random words at once. And `loss.backward()` works out, for every one of the 30,000-odd numbers
in the two tables, which way to nudge it, which is what Lab 07 opens up.

## 4. What 25,000 words were enough to learn

Gullhaven and Saltmoor are two of Kittiwake's ten towns. They appear together in only **one** ticket,
and nothing in the data says which words are towns. Predict: of the five words nearest to Gullhaven after
training, how many will be (part of) a town's name?

In [ ]:
guess("towns_cluster", None)   # a number from 0 to 5

In [ ]:
for w in ["gullhaven", "saltmoor", "nimbus", "flex"]:
    print(f"{w:10}", nearest(E, vocab, E[index[w]], 5, exclude=(w,)))
town_words = {"gullhaven", "port", "ember", "saltmoor", "cragwell", "marrowby", "eastholm", "quay",
              "wrenfield", "tern", "harbour", "lowmere", "brackenridge"}
top = [n for n, _ in nearest(E, vocab, E[index["gullhaven"]], 5, exclude=("gullhaven",))]
reveal("towns_cluster", sum(w in town_words for w in top))
print("town words among Gullhaven's five nearest:", [w for w in top if w in town_words])

All five: Brackenridge, Tern (Harbour), Lowmere, Eastholm (Quay) and Port (Ember), and Saltmoor's own
neighbours are towns too. And Nimbus's neighbours are the other handsets and their model names; Flex's are the plan
names and their numbers. The model was never told what a town is. It learned that Gullhaven and
Brackenridge are the same kind of thing because they share contexts, "my calls keep dropping in ...",
"no signal in ...", even though they almost never share a ticket. Similarity from shared neighbours rather
than from appearing together is called **second-order** co-occurrence, and it is what makes these vectors
more than a count.

Now the word that GloVe got wrong for Kittiwake:

In [ ]:
print("bill ->", nearest(E, vocab, E[index["bill"]], 6, exclude=("bill",)))

Numbers and words about paying: in Kittiwake's text, a bill is an amount of money, never a law. Same
word, same algorithm, different corpus, different meaning. Your own vectors know nothing about the world
and everything about your data; borrowed ones know the world and nothing about your data. Which you want
depends on the job, and modern systems usually start from borrowed and adapt.

## 5. The other architecture

**CBOW**, continuous bag of words, turns skip-gram around: it averages the vectors of the context words
and predicts the centre word. It trains faster, because each window is one prediction rather than
several, and it smooths over rare words; skip-gram does better with rare words and small data, which is
why this notebook uses it. The code differs in one place: the score would be the dot product of the
centre word's context row with the **average** of the surrounding words' centre rows.

## 6. Save and check

The save cell writes your pair counts for windows of 1 and 3, from your `make_pairs`, and the trained
vectors.

In [ ]:
os.makedirs("out", exist_ok=True)
json.dump({"window_1": len(make_pairs(sentences, index, 1)), "window_3": len(make_pairs(sentences, index, 3))},
          open("out/05_02_pairs.json", "w"))
np.savez("out/05_02_vectors.npz", vocab=np.array(vocab), vectors=E)
check_05_02()

## 7. Exit ticket

Explain it back: Gullhaven and Brackenridge almost never appear in the same ticket. Why do they end up
next to each other anyway? One or two sentences.

*Your explanation:* 